In [2]:
import numpy as np
import viser
import open3d as o3d

# --------------------------------------------------
# Chargement du point cloud
# Option 1 : sparse COLMAP (plus léger)
# Option 2 : gemsplat checkpoint (plus dense)
# --------------------------------------------------

# Option 1 — sparse_pc.ply COLMAP
PLY_PATH = "/home/erwinpi/data/FiGS-Standalone/3dgs/workspace/flightroom_ssv_exp/sparse_pc.ply"

pcd  = o3d.io.read_point_cloud(PLY_PATH)
pts  = np.asarray(pcd.points).astype(np.float32)
cols = np.asarray(pcd.colors).astype(np.float32)  # [0,1]

# Si pas de couleurs, mettre blanc
if cols.shape[0] == 0:
    cols = np.ones((pts.shape[0], 3), dtype=np.float32)

print(f"✅ {pts.shape[0]} points chargés")
print(f"   X: [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}]")
print(f"   Y: [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}]")
print(f"   Z: [{pts[:,2].min():.2f}, {pts[:,2].max():.2f}]")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
✅ 35930 points chargés
   X: [-78.97, 49.23]
   Y: [-31.86, 31.10]
   Z: [-11.36, 12.72]


In [3]:
server = viser.ViserServer(host="0.0.0.0", port=8765)

objects = [
    {"name": "green clock",                                 "pose": [-1.34455659,  0.76611269, -1.1]},
    {"name": "green and pink leafblower",                   "pose": [ 0.69632185,  0.01676994, -1.1]},
    {"name": "yellow handheld cordless drill on two boxes", "pose": [ 0.87418868,  1.92186469, -1.1]},
]

# Mapping axes → Viser (Y-up)
pts_viz = np.stack([
    pts[:, 0],    # X -> X
    pts[:, 2],    # Z -> Y (vertical)
    -pts[:, 1],   # -Y -> Z (profondeur)
], axis=1)

server.scene.add_point_cloud(
    name="/scene",
    points=pts_viz,
    colors=(cols * 255).astype(np.uint8),
    point_size=0.02,
)

colors_obj = [
    (0,   220, 100),
    (255, 100,   0),
    (255, 220,   0),
]

for obj, color in zip(objects, colors_obj):
    px, py, pz = obj["pose"]
    vx, vy, vz = px, pz, -py

    server.scene.add_label(
        name=f"/obj/label/{obj['name']}",
        text=obj["name"],
        position=(vx, vy + 0.15, vz),
    )
    server.scene.add_icosphere(
        name=f"/obj/sphere/{obj['name']}",
        radius=0.08,
        position=(vx, vy, vz),
        color=color,
    )

print("✅ Viser lancé → http://localhost:8765")
print("   Sur ton Mac : ssh -L 8765:localhost:8765 erwinpi@coruscant")

╭─────────────── viser ───────────────╮
│             ╷                       │
│   HTTP      │ http://0.0.0.0:8773   │
│   Websocket │ ws://0.0.0.0:8773     │
│             ╵                       │
╰─────────────────────────────────────╯

✅ Viser lancé → http://localhost:8765
   Sur ton Mac : ssh -L 8765:localhost:8765 erwinpi@coruscant
